# 01 — Data Verification

Validates the harvested PubMed corpus before publishing. Six independent checks:

1. **Inventory** — file count, total rows, date coverage, schema consistency.
2. **Integrity** — duplicate PMIDs (within and across files), null or empty fields, malformed rows.
3. **Date sanity** — precision breakdown, records bucketed outside their own pubdate year.
4. **Cross-check vs PubMed** — per-month saved count vs PubMed's live esearch count.
5. **Copy-paste queries** — exact PubMed Advanced-Search strings for manual verification of any month.
6. **Count reconciliation** — quantifies the imprecise-date bucketing that explains why the per-month sum exceeds a single all-time count.

Run top to bottom, or jump to a specific check. Read-only: the data is never modified.

## 0. Config

In [ ]:
import os, re, json, glob, time, random, collections, statistics
from datetime import date

# Resolve project root from the notebook location (works from notebooks/ or project root).
def _find_root():
    cwd = os.getcwd()
    if os.path.basename(cwd) == "notebooks":
        return os.path.dirname(cwd)
    if os.path.isdir(os.path.join(cwd, "data")) or os.path.isdir(os.path.join(cwd, "notebooks")):
        return cwd
    p = cwd
    for _ in range(5):
        if os.path.isdir(os.path.join(p, "data")):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            break
        p = parent
    return cwd

ROOT          = _find_root()
RAW_DATA_DIR  = os.path.join(ROOT, "data", "0_raw", "results")
ERROR_DIR     = os.path.join(ROOT, "data", "0_raw", "errors")
PROGRESS_FILE = os.path.join(ROOT, "data", "0_raw", "progress.json")

# Must match notebook 00 BASE_QUERY for the cross-check (Check 4) to be valid.
BASE_QUERY = (
    "english[Language]"
    " AND (USA[Affiliation] OR US[Affiliation])"
    " AND hasabstract[text]"
    " AND humans[Filter]"
    " AND Journal Article[pt]"
)

# For the live PubMed cross-check (Check 4). Optional but recommended.
API_KEY = None          # an API key string raises the rate limit; None works at 3 req/s
EMAIL   = "set-a-real-email@example.com"

files = sorted(glob.glob(os.path.join(RAW_DATA_DIR, "results_*.jsonl")))
print(f"root: {ROOT}")
print(f"found {len(files)} monthly files")
assert files, f"no results_*.jsonl found in {RAW_DATA_DIR}"

## 1. Inventory

Counts every row, records per-month totals, and checks that all files share the same schema (set of JSON keys). A schema mismatch usually indicates a code change part-way through the harvest.

In [ ]:
def month_key(path):
    m = re.search(r"results_(\d{4})_(\d{2})\.jsonl$", path)
    return f"{m.group(1)}-{m.group(2)}" if m else os.path.basename(path)

per_month = {}          # "YYYY-MM" -> row count
all_keys = collections.Counter()
schema_per_file = {}
bad_lines = []          # (file, lineno) of unparseable rows
total = 0

for fp in files:
    k = month_key(fp); n = 0; keys_here = None
    with open(fp, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                bad_lines.append((os.path.basename(fp), i))
                continue
            n += 1; total += 1
            ks = frozenset(obj.keys())
            all_keys.update(ks)
            keys_here = ks if keys_here is None else keys_here
    per_month[k] = n
    if keys_here is not None:
        schema_per_file[k] = keys_here

print(f"TOTAL ROWS: {total:,} across {len(files)} files")
print(f"date span: {min(per_month)} .. {max(per_month)}")
print(f"\nmalformed (unparseable) lines: {len(bad_lines)}")
if bad_lines:
    print("  first few:", bad_lines[:5])

# Schema consistency
schemas = set(schema_per_file.values())
if len(schemas) == 1:
    print(f"\nschema: consistent across all files — {sorted(next(iter(schemas)))}")
else:
    print(f"\nWARNING: {len(schemas)} different schemas across files!")
    base = collections.Counter(schema_per_file.values()).most_common(1)[0][0]
    for k, s in schema_per_file.items():
        if s != base:
            print(f"  {k}: extra={sorted(s-base)} missing={sorted(base-s)}")

# Missing months in the range (should be none if every month completed)
years = sorted({int(k[:4]) for k in per_month})
expected = {f"{y}-{m:02d}" for y in range(years[0], years[-1]+1) for m in range(1, 13)}
gap_months = sorted(expected - set(per_month))
print(f"\nmonths present: {len(per_month)} ; missing month files: {len(gap_months)}")
if gap_months:
    print("  missing:", gap_months[:24], "..." if len(gap_months) > 24 else "")

## 2. Integrity — duplicates and empty fields

- **Cross-file duplicate PMIDs**: the same article appearing in two months. Each PMID belongs to one PDAT bucket, so this should be near zero; a handful can occur when a record's date spans a boundary, but large numbers indicate a problem.
- **Empty critical fields**: rows missing uid, title, or abstract text.

In [ ]:
seen = {}               # uid -> first month it appeared in
cross_dups = []         # (uid, month_a, month_b)
empty = collections.Counter()
abstract_lens = []

for fp in files:
    k = month_key(fp)
    with open(fp, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                o = json.loads(line)
            except json.JSONDecodeError:
                continue
            uid = o.get("uid", "")
            if not uid:
                empty["uid"] += 1
            elif uid in seen:
                if seen[uid] != k:
                    cross_dups.append((uid, seen[uid], k))
            else:
                seen[uid] = k
            if not (o.get("title") or "").strip():
                empty["title"] += 1
            abss = o.get("abstract_sections") or []
            txt = " ".join(s.get("text", "") for s in abss).strip() if isinstance(abss, list) else ""
            if not txt:
                empty["abstract"] += 1
            else:
                abstract_lens.append(len(txt))

print(f"unique PMIDs: {len(seen):,}")
print(f"cross-file duplicate PMIDs: {len(cross_dups):,}")
if cross_dups:
    print("  examples:", cross_dups[:5])
print(f"\nempty-field counts: {dict(empty)}")
if abstract_lens:
    print(f"\nabstract length (chars): "
          f"min={min(abstract_lens)}, median={int(statistics.median(abstract_lens))}, "
          f"max={max(abstract_lens)}, mean={int(statistics.mean(abstract_lens))}")

## 3. Date sanity

Breakdown of `pubdate_precision`, plus a check for records whose parsed `pubdate` year does not match the month-file they are stored in. Some mismatch is expected — `[PDAT]` buckets electronic-ahead-of-print and imprecise dates differently from the parsed `PubDate` — but the magnitude shows how much the bucketing diverges from the literal publication date.

In [ ]:
precision = collections.Counter()
year_mismatch = 0           # file-year != pubdate-year
checked = 0

for fp in files:
    file_year = int(month_key(fp)[:4])
    with open(fp, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                o = json.loads(line)
            except json.JSONDecodeError:
                continue
            precision[o.get("pubdate_precision", "")] += 1
            pd = o.get("pubdate", "")
            if pd[:4].isdigit():
                checked += 1
                if int(pd[:4]) != file_year:
                    year_mismatch += 1

print("pubdate_precision breakdown:")
tot = sum(precision.values())
for k, v in precision.most_common():
    print(f"  {k or '(empty)':12} {v:>10,}  ({v/tot*100:5.1f}%)")
print(f"\nrecords whose pubdate-year != file-year: {year_mismatch:,} of {checked:,} "
      f"({year_mismatch/max(checked,1)*100:.2f}%)")
print("  (a small percentage is normal: electronic-vs-print dates and imprecise dates land in [PDAT] buckets)")

## 4. Cross-check vs live PubMed

For a sample of months, query PubMed for the number of records the query returns now and compare it to the number saved on disk. The gap should be small and positive (deleted or withheld records, plus drift since the harvest). This is the direct test of whether the counts are correct.

Set `SAMPLE_MONTHS` to a handful of months (more = slower). Requires internet. Uses the API key from Check 0 if set.

In [ ]:
import requests

SAMPLE_MONTHS = ["1994-01", "2000-01", "2009-01", "2015-06", "2020-04", "2024-12"]  # edit as needed

def pubmed_count(term):
    p = {"db": "pubmed", "term": term, "retmode": "json", "retmax": 0,
         "tool": "verify", "email": EMAIL}
    if API_KEY:
        p["api_key"] = API_KEY
    r = requests.get("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
                     params=p, timeout=60)
    r.raise_for_status()
    return int(r.json()["esearchresult"]["count"])

print(f"{'month':9} {'saved':>9} {'pubmed':>9} {'gap':>7} {'gap%':>7}  status")
print("-" * 55)
for k in SAMPLE_MONTHS:
    if k not in per_month:
        print(f"{k:9} (no file)"); continue
    y, m = k.split("-")
    term = f"{BASE_QUERY} AND {y}/{m}[PDAT]"
    try:
        pc = pubmed_count(term)
    except Exception as e:
        print(f"{k:9} query failed: {e}"); continue
    s = per_month[k]; gap = pc - s; rate = gap / pc * 100 if pc else 0
    status = "OK" if abs(rate) <= 2 else "CHECK"
    print(f"{k:9} {s:>9,} {pc:>9,} {gap:>7,} {rate:>6.2f}%  {status}")
    time.sleep(0.4 if not API_KEY else 0.12)

## 5. Copy-paste PubMed queries

Prints exact strings for PubMed Advanced Search (https://pubmed.ncbi.nlm.nih.gov/advanced/) for manual verification of any month. The count PubMed shows should match the "pubmed" column above, and sit a hair above the saved count.

In [ ]:
def pubmed_query_for(month_key_str):
    y, m = month_key_str.split("-")
    return f"{BASE_QUERY} AND {y}/{m}[PDAT]"

print("Whole-query (no date) — compare to the grand total, allowing for the date-bucketing note below:")
print(f"  {BASE_QUERY}\n")
print("Per-month (paste any of these):")
for k in ["1994-01", "2009-01", "2020-04", "2024-12"]:
    print(f"  {k}:  {pubmed_query_for(k)}")
print("\nA single high-volume day (the Jan-1 pile-up), e.g.:")
print(f"  {BASE_QUERY} AND 2009/01/01[PDAT]")

## 6. Count reconciliation

A single all-time count of the query (no date restriction) is lower than the sum of per-month counts. This is expected: records with **imprecise dates** (year-only or year+month) are bucketed into months by `[PDAT]`, and the sum of 384 monthly slices counts each one in its bucket. The cell below quantifies how much of the corpus has imprecise dates — that is the bulk of the difference.

In [ ]:
imprecise = precision.get("year", 0) + precision.get("year_month", 0)
precise   = precision.get("full_date", 0)
print(f"full_date (precise):    {precise:>10,}  ({precise/tot*100:4.1f}%)")
print(f"year_month (imprecise): {precision.get('year_month',0):>10,}  ({precision.get('year_month',0)/tot*100:4.1f}%)")
print(f"year only (imprecise):  {precision.get('year',0):>10,}  ({precision.get('year',0)/tot*100:4.1f}%)")
print(f"\n{imprecise:,} of {tot:,} records ({imprecise/tot*100:.0f}%) have imprecise dates.")
print("These are bucketed by [PDAT] (often onto the 1st of a month or year), which is why the")
print("sum of per-month counts exceeds a single unrestricted all-time count.")
print("\nTo reconcile against a single all-time count, run the whole-query (Check 5) and note that")
print("the live count drifts daily as PubMed reindexes.")

## 7. Harvest error and missing logs

Reads `errors/missing_*.json` (deleted or withheld counts recorded during the harvest) and flags any month above the 2% threshold — those are worth re-running; the rest are normal. Also reports any leftover `failed_*` piece files, which indicate a piece that never completed.

In [ ]:
miss = sorted(glob.glob(os.path.join(ERROR_DIR, "missing_*.json")))
total_missing = 0; flagged = []
for mf in miss:
    try:
        d = json.load(open(mf, encoding="utf-8"))
    except Exception:
        continue
    total_missing += d.get("missing_count", 0)
    if d.get("missing_rate", 0) > 0.02:
        flagged.append((d.get("month"), d.get("missing_rate")))
print(f"{total_missing:,} PMIDs not returned across {len(miss)} months (deleted or withheld)")
print(f"overall miss rate: {total_missing/max(total,1)*100:.2f}% of saved rows")
if flagged:
    print(f"\nmonths above 2% (re-run these): {flagged}")
else:
    print("\nno month exceeds the 2% threshold — all within normal deleted/withheld range.")

failed = glob.glob(os.path.join(ERROR_DIR, "failed_*.json"))
print(f"\nunresolved 'failed_*' piece files: {len(failed)}", failed[:5] if failed else "")